# T5-small Inference on Spider Dev

Loads fine-tuned checkpoint from Drive, generates SQL for all dev examples, writes `pred_t5.txt` aligned to `dev.json`.

Runs in ~5-10 min on Colab T4. Download `pred_t5.txt` and `gold.txt` at the end for local evaluation.

In [ ]:
!pip install -q transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

DRIVE_ROOT = Path('/content/drive/MyDrive/text-to-sql')
SPIDER_DIR = DRIVE_ROOT / 'spider_data'
CHECKPOINT = DRIVE_ROOT / 't5_small_spider' / 'final'
OUT_DIR = Path('/content/predictions')
OUT_DIR.mkdir(exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
def load_json(path):
    with open(path) as f:
        return json.load(f)

validation = load_json(SPIDER_DIR / 'dev.json')
tables = load_json(SPIDER_DIR / 'tables.json')

def build_schema_string(db_table_info):
    parts = []
    table_names = db_table_info['table_names_original']
    columns = db_table_info['column_names_original']
    tables_to_cols = {i: [] for i in range(len(table_names))}
    for tbl_idx, col_name in columns:
        if tbl_idx == -1:
            continue
        tables_to_cols[tbl_idx].append(col_name)
    for i, tbl in enumerate(table_names):
        cols = ', '.join(tables_to_cols[i])
        parts.append(f'{tbl}({cols})')
    return ' | '.join(parts)

schema_by_db = {t['db_id']: build_schema_string(t) for t in tables}

def make_input(ex):
    return f"translate to SQL: {ex['question']} | schema: {schema_by_db[ex['db_id']]}"

print(f'dev examples: {len(validation)}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(str(CHECKPOINT))
model = AutoModelForSeq2SeqLM.from_pretrained(str(CHECKPOINT)).to(device)
model.eval()
print('Model loaded')

In [ ]:
with open(OUT_DIR / 'gold.txt', 'w', encoding='utf-8') as f:
    for ex in validation:
        q = ' '.join(ex['query'].split())
        f.write(f"{q}\t{ex['db_id']}\n")
print('gold.txt written')

In [ ]:
BATCH_SIZE = 16
MAX_INPUT_LEN = 512
MAX_NEW_TOKENS = 256
NUM_BEAMS = 4

preds = []

with torch.no_grad():
    for start in tqdm(range(0, len(validation), BATCH_SIZE)):
        batch = validation[start:start + BATCH_SIZE]
        inputs_text = [make_input(ex) for ex in batch]
        enc = tokenizer(
            inputs_text,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(device)
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            early_stopping=True,
        )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        preds.extend(decoded)

print(f'Generated {len(preds)} predictions')

In [ ]:
with open(OUT_DIR / 'pred_t5.txt', 'w', encoding='utf-8') as f:
    for p in preds:
        p = ' '.join(p.split()).strip()
        if not p:
            p = 'SELECT 1'
        f.write(p + '\n')

with open(OUT_DIR / 'gold.txt') as f:
    n_gold = sum(1 for _ in f)
with open(OUT_DIR / 'pred_t5.txt') as f:
    n_pred = sum(1 for _ in f)

print(f'gold.txt: {n_gold} lines')
print(f'pred_t5.txt: {n_pred} lines')
assert n_gold == n_pred, 'Alignment broken'

In [ ]:
for i in range(5):
    print(f'--- example {i} ---')
    print('Q   :', validation[i]['question'])
    print('GOLD:', validation[i]['query'])
    print('PRED:', preds[i])
    print()

In [ ]:
from google.colab import files
files.download(str(OUT_DIR / 'pred_t5.txt'))
files.download(str(OUT_DIR / 'gold.txt'))